# MID-TERM PROJECT

PROJECT EXPLANATION

In [ ]:
# LOAD PACKAGES
# 0.1 - DATA PREP AND MATH LIBRARIES
import pandas as pd
import numpy as np

# PLOTTING
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline 

# SPLITTING DATASET
from sklearn.model_selection import train_test_split

# DECISION TREES
from sklearn.tree               import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics            import roc_auc_score
from sklearn.tree               import export_text

# ENSEMBLE LEARNING
from sklearn.ensemble           import RandomForestClassifier

# PYSPARK
from pyspark.sql                import functions as F
from pyspark.sql.types          import (
                                    NumericType,
                                    StringType,
                                    NullType,
                                )




In [ ]:
# 0.2 - GET DATA

data_path = "C:/Users/duffl/Documents/Chris/Training/Python/ML-Zoomcamp-2026/midterm_project/raw_data/trees_through_time_20250429.csv"
df       = pd.read_csv(data_path)

# STANDARDIZE ALL COLUMN NAMES
df.columns = df.columns.str.lower()


## PART 1 - EXPLORATORY DATA ANALYSIS

In [ ]:
# EDA

# PREVIEW COUNT OF ROWS WITH DATA BY COLUMN
print(
    f"df.count()\n"
    f"===============================\n"
    f"{df.count()}\n"
)

# IDENTIFY COLUMNS WITH MISSING VALUES
# TREAT NaN AND null AS MISSING FOR NUMERIC VALUES
missing_columns = []

for field in df.schema.fields:
    col = F.col(field.name)
    missing_condition = col.isNull()

    if isinstance(field.dataType, NumericType):
        missing_condition = missing_condition | F.isnan(col)

    if df.filter(missing_condition).limit(1).count() > 0:
        missing_columns.append(field.name)

print(
    "missing_columns (columns missing values)\n"
    "===============================\n"
    f"{missing_columns}"
)

# CREATE LISTS OF CATEGOTICAL AND NUMERIC COLUMNS FROM THE SPARK SCHEMA
categorical_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

numerical_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

# SHOW FREQUENCY OF VALUES WITHIN CATEGORICAL COLUMNS
#   EXCLUDE null VALUES
for c in categorical_columns:
    print(f"\nValue counts for {c}:")
    (
        df.groupBy(c)
          .count()
          .filter(F.col(c).isNotNull())
          .orderBy(F.desc("count"))
          .show(truncate=False)
    )

# SHOW DESCRIPTIVE STATISTICS FOR NUMERIC COLUMNS
for n in numerical_columns:
    print(f"\nDescriptive statistics for {n}:")
    df.select(n).summary(
        "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
    ).show()